# 🧑‍🍳 Neosantara AI Cookbook: OpenAI SDK Migration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/openai-to-responses-migration.ipynb)

This recipe helps you migrate from the traditional **Chat Completions API** (`v1/chat/completions`) to the modern **Responses API** (`v1/responses`). 

The Responses API is designed to be the foundational interface for building agentic applications, offering built-in state management and more natural tool handling.

In [ ]:
# Install the OpenAI SDK (v1.50.0+ required for Responses API)
!pip install "openai>=1.50.0"

# Setup API Key (for Google Colab)
try:
    from google.colab import userdata
    import os
    os.environ["NEOSANTARA_API_KEY"] = userdata.get('NEOSANTARA_API_KEY')
    print("API Key loaded from Google Colab userdata.")
except ImportError:
    import os
    if "NEOSANTARA_API_KEY" in os.environ:
        print("API Key loaded from environment variables.")
    else:
        print("Not running in Google Colab and NEOSANTARA_API_KEY not found in environment.")

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("NEOSANTARA_API_KEY"),
    base_url="https://api.neosantara.xyz/v1"
)

## 🔄 Comparison: Chat Completions vs. Responses API

### 1. Simple Prompt

#### **The Old Way (Chat Completions)**
Uses `messages` (list of objects) and requires manual parsing of the response.

In [ ]:
completion = client.chat.completions.create(
    model="nusantara-base",
    messages=[
        {"role": "user", "content": "What is the capital of Indonesia?"}
    ]
)

print(completion.choices[0].message.content)

#### **The New Way (Responses API)**
Uses `input` (string) and provides a direct `output_text` property.

In [ ]:
response = client.responses.create(
    model="nusantara-base",
    input="What is the capital of Indonesia?"
)

print(response.output_text)

## 🛠️ Complex Example: Tool Calling

The Responses API supports both built-in tools (like Web Search) and custom function calling. Unlike Chat Completions, the model can automatically execute built-in tools when available.

In [ ]:
# Example using built-in Web Search
response = client.responses.create(
    model="nusantara-base",
    tools=[{"type": "web_search"}],
    input="Who won the most recent Indonesian presidential election?"
)

print(response.output_text)

### Custom Function Calling
Defining custom tools follows a similar pattern to Chat Completions but is integrated into the `tools` list of `responses.create`.

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_ticket_price",
        "description": "Get the price of a train ticket in Indonesia",
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {"type": "string"},
                "destination": {"type": "string"}
            },
            "required": ["origin", "destination"]
        }
    }
}]

response = client.responses.create(
    model="nusantara-base",
    tools=tools,
    input="How much is a train ticket from Jakarta to Bandung?"
)

print(f"Tool call detected: {response.output_text is None}")
# If output_text is None, check response.tool_calls
if not response.output_text:
    for tool_call in response.tool_calls:
        print(f"Calling tool: {tool_call.function.name} with arguments: {tool_call.function.arguments}")